In [32]:
import numpy as np
import pandas as pd
from pathlib import Path

c:\Users\Sultan Selin\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [33]:
## merged data from parquet file 

MERGED_PATH = (
    Path.cwd().parent
    / "data"
    / "processed"
    / "merged_data.parquet"
)
data = pd.read_parquet(MERGED_PATH)
print(data["date"].dt.tz)
data.info()


Europe/Istanbul
<class 'pandas.DataFrame'>
RangeIndex: 84696 entries, 0 to 84695
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype                          
---  ------         --------------  -----                          
 0   date           84696 non-null  datetime64[us, Europe/Istanbul]
 1   consumption    84696 non-null  float64                        
 2   year           84696 non-null  int32                          
 3   month          84696 non-null  int32                          
 4   day            84696 non-null  int32                          
 5   weekday        84696 non-null  str                            
 6   national_temp  84696 non-null  float64                        
 7   hour           84696 non-null  int32                          
dtypes: datetime64[us, Europe/Istanbul](1), float64(2), int32(4), str(1)
memory usage: 4.5 MB


In [34]:
#Lags
data["lag_24"] = data["consumption"].shift(24)
data["lag_48"] = data["consumption"].shift(48)
data["lag_168"] = data["consumption"].shift(168)


#Rolling means
data["rolling_mean_24"]  = data["consumption"].shift(24).rolling(24).mean()
data["rolling_mean_168"] = data["consumption"].shift(24).rolling(168).mean()
# shift(24) before rolling: forecast horizon is 24h, so no hour
# within that window is available at prediction time


In [35]:
print(data["date"].is_monotonic_increasing)
print(data.info())
data

True
<class 'pandas.DataFrame'>
RangeIndex: 84696 entries, 0 to 84695
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype                          
---  ------            --------------  -----                          
 0   date              84696 non-null  datetime64[us, Europe/Istanbul]
 1   consumption       84696 non-null  float64                        
 2   year              84696 non-null  int32                          
 3   month             84696 non-null  int32                          
 4   day               84696 non-null  int32                          
 5   weekday           84696 non-null  str                            
 6   national_temp     84696 non-null  float64                        
 7   hour              84696 non-null  int32                          
 8   lag_24            84672 non-null  float64                        
 9   lag_48            84648 non-null  float64                        
 10  lag_168           84528 non-null  float6

,date,consumption,year,month,day,weekday,national_temp,hour,lag_24,lag_48,lag_168,rolling_mean_24,rolling_mean_168
0,2017-01-01 00:00:00+03:00,27223.06,2017,1,1,Sunday,1.314326,0,NaN,NaN,NaN,NaN,NaN
1,2017-01-01 01:00:00+03:00,25825.90,2017,1,1,Sunday,0.719622,1,NaN,NaN,NaN,NaN,NaN
2,2017-01-01 02:00:00+03:00,24252.68,2017,1,1,Sunday,0.304513,2,NaN,NaN,NaN,NaN,NaN
3,2017-01-01 03:00:00+03:00,22915.47,2017,1,1,Sunday,-0.786601,3,NaN,NaN,NaN,NaN,NaN
4,2017-01-01 04:00:00+03:00,22356.99,2017,1,1,Sunday,-0.671492,4,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
84691,2026-08-30 19:00:00+03:00,43487.20,2026,8,30,Sunday,25.720893,19,46146.69,50946.09,48107.67,43362.090833,47861.601488
84692,2026-08-30 20:00:00+03:00,44306.44,2026,8,30,Sunday,24.377825,20,46276.80,50700.57,49124.39,43177.767083,47824.966548
84693,2026-08-30 21:00:00+03:00,43046.39,2026,8,30,Sunday,24.069101,21,44800.31,49012.85,47802.28,43002.244583,47790.170952
84694,2026-08-30 22:00:00+03:00,41560.60,2026,8,30,Sunday,23.356092,22,42736.17,47010.37,46263.88,42824.152917,47754.071429


In [36]:
data["is_weekend"] = data["weekday"].isin(["Saturday", "Sunday"]).astype(int)
data["DoW"] = data["date"].dt.dayofweek
data["DoY"] = data["date"].dt.dayofyear
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 84696 entries, 0 to 84695
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype                          
---  ------            --------------  -----                          
 0   date              84696 non-null  datetime64[us, Europe/Istanbul]
 1   consumption       84696 non-null  float64                        
 2   year              84696 non-null  int32                          
 3   month             84696 non-null  int32                          
 4   day               84696 non-null  int32                          
 5   weekday           84696 non-null  str                            
 6   national_temp     84696 non-null  float64                        
 7   hour              84696 non-null  int32                          
 8   lag_24            84672 non-null  float64                        
 9   lag_48            84648 non-null  float64                        
 10  lag_168           84528 non-null  float64    

In [ ]:
from datetime import date
import holidays
tr_holidays = holidays.TR(years=range(2017, 2027))

from datetime import timedelta
holiday_set = set(tr_holidays.keys()) ### gives dates of the holidays

def is_arife(date):
    tomorrow = (date + timedelta(days=1)).date()
    today = date.date()
    if tomorrow in holiday_set and today not in holiday_set:
        name = tr_holidays[tomorrow]
        return "Kurban Bayramı" in name or "Ramazan Bayramı" in name
    return False
data["is_arife"] = data["date"].apply(is_arife)
data  ## is_arife True count is 456 which a day is missing


,date,consumption,year,month,day,weekday,national_temp,hour,lag_24,lag_48,lag_168,rolling_mean_24,rolling_mean_168,is_weekend,DoW,DoY,is_arife
0,2017-01-01 00:00:00+03:00,27223.06,2017,1,1,Sunday,1.314326,0,NaN,NaN,NaN,NaN,NaN,1,6,1,False
1,2017-01-01 01:00:00+03:00,25825.90,2017,1,1,Sunday,0.719622,1,NaN,NaN,NaN,NaN,NaN,1,6,1,False
2,2017-01-01 02:00:00+03:00,24252.68,2017,1,1,Sunday,0.304513,2,NaN,NaN,NaN,NaN,NaN,1,6,1,False
3,2017-01-01 03:00:00+03:00,22915.47,2017,1,1,Sunday,-0.786601,3,NaN,NaN,NaN,NaN,NaN,1,6,1,False
4,2017-01-01 04:00:00+03:00,22356.99,2017,1,1,Sunday,-0.671492,4,NaN,NaN,NaN,NaN,NaN,1,6,1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84691,2026-08-30 19:00:00+03:00,43487.20,2026,8,30,Sunday,25.720893,19,46146.69,50946.09,48107.67,43362.090833,47861.601488,1,6,242,False
84692,2026-08-30 20:00:00+03:00,44306.44,2026,8,30,Sunday,24.377825,20,46276.80,50700.57,49124.39,43177.767083,47824.966548,1,6,242,False
84693,2026-08-30 21:00:00+03:00,43046.39,2026,8,30,Sunday,24.069101,21,44800.31,49012.85,47802.28,43002.244583,47790.170952,1,6,242,False
84694,2026-08-30 22:00:00+03:00,41560.60,2026,8,30,Sunday,23.356092,22,42736.17,47010.37,46263.88,42824.152917,47754.071429,1,6,242,False


In [ ]:
#date(2026, 5, 27) in tr_holidays ----returns True or False
#tr_holidays.get('2026-05-27') ---------name
#tr_holidays['2026-05-27']   -----name

True

In [ ]:

data[data["is_arife"] == True].groupby(data["date"].dt.year)["is_arife"].sum() / 24


date
2017    2.0
2018    2.0
2019    2.0
2020    2.0
2021    2.0
2022    1.0
2023    2.0
2024    2.0
2025    2.0
2026    2.0
Name: is_arife, dtype: float64